In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
train = pd.read_parquet(r"C:\Users\Gulshat\Downloads\NLP_avito_interns\NLP_avito_interns\dataset\train.parquet")
queris = pd.read_parquet(r"C:\Users\Gulshat\Downloads\NLP_avito_interns\NLP_avito_interns\dataset\benchmark_queries.parquet")
items = pd.read_parquet(r"C:\Users\Gulshat\Downloads\NLP_avito_interns\NLP_avito_interns\dataset\benchmark_items.parquet")

In [3]:
train.shape


(497673, 19)

In [4]:
print(f'Train(обучающая выборка): {train.shape}')
print(f'Queris(запросы для поиска): {queris.shape}')
print(f'Items(корпус объявлений): {items.shape}')

Train(обучающая выборка): (497673, 19)
Queris(запросы для поиска): (2452, 6)
Items(корпус объявлений): (189212, 14)


In [5]:
train.head()

,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
0,скупка телевизоров,652430,0,,114,Скупка б/у техники,91.0,4.989011,1.000000000000000,2303374,39.712818145751953,652430,54.629230499267578,True,False,Вид услуги Место оказания услуг Первомайский п...,e8b685dffe1a408e,Скупаю практически любую современную новую и б...,114
1,автоподбор,640860,0,Рейтинг пользователя 4 звезды и выше,114,Автоподбор Разовый осмотр автомобиля,462.0,4.982684,3500.000000000000000,2303374,44.051009999999998,640860,56.273389999999999,False,False,Вид услуги Место оказания услуг Нижний Новгоро...,92e1b0446f827b59,🚗 Автоподбор и выездная диагностика автомобиля...,114
2,баня на дровах,653240,0,"Онлайн-запись Тип услуги СПА-услуги, массаж Ви...",114,"Баня на дровах ""Прованс"" на Цветочной",NaN,NaN,1600.000000000000000,86469,30.128784180000000,653240,59.783687590000000,False,False,"Вид услуги Красота, здоровье Место оказания ус...",624846856ce81d69,В ритме современной жизни так сложно найти мо...,114
3,изготовление госномера на авто,634670,0,"Вид услуги Оборудование, производство",114,"Изготовление дубликатов авто номеров, гос номеров",14.0,4.714286,1700.000000000000000,2303428,40.537841000000000,633570,45.424875000000000,False,False,"Вид услуги Оборудование, производство Тип услу...",45b8628b9c6e7b85,Изготовим дубликат номера по утере или износу ...,114
4,укладка плитки,658430,0,Тип услуги Ремонт квартир и домов под ключ Вид...,114,Ремонт и отделка квартир под ключ,1.0,5.000000,1000.000000000000000,44725,69.496444699999998,658430,56.105983729999998,False,False,Вид услуги Ремонт и отделка Тип услуги Ремонт ...,c95a4a7daf2a967f,отделочные работы любой сложности.,114


In [6]:
queris.head()

,query_id,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category
0,70DfDUpwjxB4lzFd,перевозки владикавказ тбилиси,649820,0,,114
1,JTrdTaZJvSiLPkXj,обзвон по базе,107620,0,Вид услуги Деловые услуги,114
2,LZCZNoVG4AFUkVRJ,липоредукция подбородка,637640,0,"Вид услуги Красота, здоровье",114
3,660ac9QVtXkRxZC3,подъемник 4 х стоечный,662810,0,,114
4,YgHcM9MVbxKnxD1e,монтаж видеодомофонов,642790,0,,114


In [7]:
items.head()

,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
0,Ремонт/выкуп компьют. и ноутбуков с выездом на...,57.0,5.0,500.000000000000000,2097573,42.047193961004901,631060,44.228180450924000,False,False,Вид услуги Компьютерная помощь Место оказания ...,111eb8b979577d79,"Выезд на дом в любое время, вплоть до 23:00\n\...",114
1,Обучение ребенка чтению,7.0,5.0,900.000000000000000,86453,49.627391820000000,631870,58.627609249999999,False,False,"Вид услуги Обучение, курсы Место оказания услу...",75fc8e10f5a66fc4,"Дорогие родители и маленькие книголюбы! \n\nЯ,...",114
2,Афрокудри 5+,14.0,5.0,1000.000000000000000,86467,40.936892000000000,628500,56.990307000000001,True,False,"Вид услуги Красота, здоровье Место оказания ус...",3f6ae81704565b9c,ВНИМАНИЕ Укладка !!!! 🥳\n\nАФРОкудри отличный ...,114
3,Монтаж малых архитектурных форм,2.0,5.0,5000.000000000000000,2058739,37.576183000000000,637640,55.662734999999998,False,False,Вид услуги Строительство Место оказания услуг ...,0618dec37a59a21a,Оказываем все виды услуг по сборке и монтажу и...,114
4,Репетитор по истории 10 класс ЕГЭ,19.0,5.0,1500.000000000000000,86456,44.750473999999997,624850,48.786008000000002,False,False,"Вид услуги Обучение, курсы Место оказания услу...",13da2c81574677ed,🚀 Сдай ЕГЭ по истории на 80+ за 6 месяцев!\n\n...,114


In [8]:
# Заполняем пропуски пустыми строками
# Список колонок для запросов
query_cols = ['search_query', 'search_infm_params_text']
for col in query_cols:
    queris[col] = queris[col].fillna('')

# Список колонок для объявлений
item_cols = ['item_title_raw', 'item_description_raw', 'item_infm_params_text']
for col in item_cols:
    items[col] = items[col].fillna('')

# Склеиваем текстовые признаки в один столбец 'text'
queris['text'] = queris['search_query'] + ' ' + queris['search_infm_params_text']
items['text'] = items['item_title_raw'] + ' ' + items['item_description_raw'] + ' ' + items['item_infm_params_text']

# Инициализируем TF-IDF
vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))

# Обучаем векторизатор на текстах объявлений и преобразуем их
item_matrix = vectorizer.fit_transform(items['text'])

# Преобразуем тексты запросов, используя уже обученный словарь
query_matrix = vectorizer.transform(queris['text'])

print(f"Размер матрицы объявлений: {item_matrix.shape}")
print(f"Размер матрицы запросов: {query_matrix.shape}")

Размер матрицы объявлений: (189212, 50000)
Размер матрицы запросов: (2452, 50000)


In [10]:
# Параметры поиска
top_n = 50
batch_size = 200
item_ids = items['item_id'].values
all_predictions = []

# Итерируемся по запросам порциями, чтобы не перегружать память
for start_idx in range(0, query_matrix.shape[0], batch_size):
    end_idx = min(start_idx + batch_size, query_matrix.shape[0])
    
    # Вырезаем текущую порцию запросов
    batch_query = query_matrix[start_idx:end_idx]
    
    # Считаем косинусное сходство для этой порции
    similarities = (batch_query @ item_matrix.T).toarray()
    
    # Получаем индексы топ-50 (сортируем по убыванию)
    top_indices = np.argsort(similarities, axis=1)[:, -top_n:][:, ::-1]
    
    # Сохраняем item_id для каждого запроса
    for row in top_indices:
        all_predictions.append(item_ids[row].tolist())
        
    print(f"Обработано: {end_idx} / {query_matrix.shape[0]}")

# Формируем итоговый датафрейм
answer = pd.DataFrame({
    'query_id': queris['query_id'],
    'answer': [' '.join(map(str, top50)) for top50 in all_predictions]
})

# Сохраняем результат в файл
answer.to_csv('answer.csv', index=False, encoding='utf-8')

print("Готово. Размер файла:", answer.shape)

Обработано: 200 / 2452
Обработано: 400 / 2452
Обработано: 600 / 2452
Обработано: 800 / 2452
Обработано: 1000 / 2452
Обработано: 1200 / 2452
Обработано: 1400 / 2452
Обработано: 1600 / 2452
Обработано: 1800 / 2452
Обработано: 2000 / 2452
Обработано: 2200 / 2452
Обработано: 2400 / 2452
Обработано: 2452 / 2452
Готово. Размер файла: (2452, 2)


In [11]:
pd.read_csv('answer.csv')

,query_id,answer
0,70DfDUpwjxB4lzFd,355392014208b7bf 4332513c8fe7e1c9 9515d1e1ecda...
1,JTrdTaZJvSiLPkXj,cbeccbecb1fb8d86 664665688c35c1d0 58616767e88c...
2,LZCZNoVG4AFUkVRJ,0b1c884e6c242ea7 168a9207e80b0be4 7347ccc4bdcf...
3,660ac9QVtXkRxZC3,88fbcdb287ccfe3f 41ad6e1ba0d270a8 c9f23facbc1b...
4,YgHcM9MVbxKnxD1e,705d96426f3ad807 683aca7577a9f5ee 8f1645bec9c6...
...,...,...
2447,JYUbsR4EU5RtfGUA,51143556cf7d99fc 66ccc7c49ba98057 ddbac35a8116...
2448,SuCUWvKrKFRKTKuT,2330db10aebf8fc0 fb0bf6077c59c5a2 d65c4333d12e...
2449,fWIRLM5wCElVVjRV,8ae2b3466215cf32 b0f353237853ac7c 18c330f57fbd...
2450,VjX4ayHpKHlftRgu,59822dfd3e9ec127 5d8e3434c258213b 8646ef5e1187...
